# 🦙 LEVEL 2 — LlamaIndex Data Loading

**Topics:** SimpleDirectoryReader, Document metadata, CSV/JSON loading, Custom BaseReader, IngestionPipeline, TechNova Mini-Project

Prerequisites: Level 1 complete, `data/level2_samples/` created.

In [ ]:
# STEP 0: Settings — ALWAYS FIRST
import os
from dotenv import load_dotenv
load_dotenv()

from llama_index.core import Settings
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding

Settings.llm = GoogleGenAI(model='gemini-2.0-flash', api_key=os.getenv('GOOGLE_API_KEY'))
Settings.embed_model = GoogleGenAIEmbedding(model_name='models/text-embedding-004', api_key=os.getenv('GOOGLE_API_KEY'))
Settings.chunk_size = 512
Settings.chunk_overlap = 50
print('✅ Settings configured')

## Topic 17 — Document Deep Dive

In [ ]:
from llama_index.core import Document

doc = Document(
    text='Q4 2024 Revenue: $7.175M (86% YoY). EBITDA: $1.945M. New customers: 47.',
    doc_id='technova-q4-2024-financial',
    metadata={
        'source': 'quarterly_report_q4_2024.pdf',
        'doc_type': 'financial_report',
        'quarter': 'Q4', 'year': '2024',
        'department': 'Finance', 'company': 'TechNova Inc.',
        'author': 'Finance Team',        # exclude from embed
        'internal_id': 'FIN-2024-Q4-001', # exclude from LLM
    },
    excluded_embed_metadata_keys=['author', 'internal_id'],
    excluded_llm_metadata_keys=['internal_id'],
)

print(f'doc_id    : {doc.doc_id}')
print(f'metadata  : {doc.metadata}')
print(f'embed excl: {doc.excluded_embed_metadata_keys}')
print(f'llm excl  : {doc.excluded_llm_metadata_keys}')
print()
print('LLM Context preview:')
print(doc.get_content(metadata_mode='all'))

## Topic 19 — SimpleDirectoryReader

In [ ]:
from llama_index.core import SimpleDirectoryReader
from datetime import datetime

docs = SimpleDirectoryReader(
    input_dir='data/level2_samples',
    required_exts=['.txt', '.md'],
    recursive=True,
    filename_as_id=True,
    file_metadata=lambda fp: {
        'company': 'TechNova Inc.',
        'project': 'technova_kb',
        'environment': 'dev',
        'ingested_at': datetime.now().isoformat(),
    }
).load_data()

print(f'Loaded {len(docs)} text/markdown documents')
for d in docs:
    print(f'  [{d.doc_id}]')
    print(f'  file    : {d.metadata.get("file_name")}')
    print(f'  project : {d.metadata.get("project")}')
    print()

## Topic 21 — CSV Loading (Row-Level Documents)

In [ ]:
import pandas as pd
from llama_index.core import Document

df = pd.read_csv('data/level2_samples/employees.csv')
print(f'CSV: {df.shape[0]} rows, columns: {df.columns.tolist()}')

employee_docs = []
for _, row in df.iterrows():
    text = (f'Employee: {row["name"]}\n'
            f'Dept    : {row["department"]}\n'
            f'Role    : {row["role"]}\n'
            f'Location: {row["location"]}\n'
            f'Score   : {row["performance_score"]}/5.0')
    employee_docs.append(Document(
        text=text,
        doc_id=f'employee-{row["employee_id"]}',
        metadata={
            'employee_id': row['employee_id'],
            'name': row['name'],
            'department': row['department'],
            'role': row['role'],
            'salary_band': row['salary_band'],
            'location': row['location'],
            'performance_score': str(row['performance_score']),
            'doc_type': 'employee_record',
            'source': 'employees.csv',
        },
        excluded_embed_metadata_keys=['employee_id', 'salary_band'],
        excluded_llm_metadata_keys=['salary_band', 'employee_id'],
    ))

print(f'Created {len(employee_docs)} employee Documents')
for d in employee_docs[:3]:
    print(f'  [{d.doc_id}] {d.metadata["department"]} | {d.text[:60].strip()}...')

## Topic 22 — JSON Loading (Section-Level Documents)

In [ ]:
import json
from llama_index.core import Document

with open('data/level2_samples/quarterly_report.json') as f:
    data = json.load(f)

fin = data['financials']
met = data['key_metrics']
cm = {'source': 'quarterly_report.json', 'quarter': data['quarter'],
      'year': str(data['year']), 'company': data['company'], 'department': 'Finance'}

json_docs = [
    Document(
        text=(f"{data['quarter']} {data['year']} Financials:\n"
              f"Revenue: ${fin['revenue']['total']:,}\n"
              f"EBITDA : ${fin['ebitda']:,}\n"
              f"Net Income: ${fin['net_income']:,}\n"
              f"Gross Margin: {fin['gross_margin_pct']}%"),
        doc_id=f"technova-{data['quarter']}-{data['year']}-financials",
        metadata={**cm, 'doc_type': 'financial_report', 'section': 'financials'},
    ),
    Document(
        text=(f"{data['quarter']} {data['year']} Key Metrics:\n"
              f"New Customers: {met['new_customers']}\n"
              f"Churn Rate   : {met['churn_rate_pct']}%\n"
              f"NPS Score    : {met['nps_score']}\n"
              f"ARR Growth   : {met['arr_growth_pct']}%"),
        doc_id=f"technova-{data['quarter']}-{data['year']}-metrics",
        metadata={**cm, 'doc_type': 'business_metrics', 'section': 'key_metrics'},
    ),
    Document(
        text='Q4 2024 Highlights:\n' + '\n'.join(f'• {h}' for h in data['highlights']),
        doc_id=f"technova-{data['quarter']}-{data['year']}-highlights",
        metadata={**cm, 'doc_type': 'highlights', 'section': 'highlights'},
    ),
]

print(f'Created {len(json_docs)} JSON section Documents')
for d in json_docs:
    print(f'  [{d.doc_id}] — section={d.metadata["section"]} | {len(d.text)} chars')

## Topic 27 — Custom BaseReader

In [ ]:
from llama_index.core.readers.base import BaseReader
from llama_index.core import Document
from typing import List, Optional, Dict, Any
from pathlib import Path
import csv

class CSVRowReader(BaseReader):
    """Loads CSV: one Document per row. text_columns → Document text, metadata_columns → metadata."""
    def __init__(self, text_columns, metadata_columns, id_column=None):
        self.text_columns = text_columns
        self.metadata_columns = metadata_columns
        self.id_column = id_column
    
    def load_data(self, file: Path, extra_info=None) -> List[Document]:
        documents = []
        with open(file, newline='', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            for i, row in enumerate(reader):
                text = '\n'.join(
                    f'{col.replace("_", " ").title()}: {row.get(col, "")}'
                    for col in self.text_columns if col in row
                )
                metadata = {col: row.get(col, '') for col in self.metadata_columns}
                metadata['source'] = file.name
                metadata['doc_type'] = 'csv_row'
                if extra_info:
                    metadata.update(extra_info)
                doc_id = (f'{file.stem}-{row[self.id_column]}'
                         if self.id_column and self.id_column in row
                         else f'{file.stem}-row-{i}')
                documents.append(Document(text=text, doc_id=doc_id, metadata=metadata))
        return documents

reader = CSVRowReader(
    text_columns=['name', 'department', 'role', 'location'],
    metadata_columns=['employee_id', 'department', 'role', 'location', 'performance_score'],
    id_column='employee_id'
)
docs = reader.load_data(Path('data/level2_samples/employees.csv'), extra_info={'company': 'TechNova'})
print(f'Custom CSVRowReader: {len(docs)} Documents')
for d in docs[:3]:
    print(f'  [{d.doc_id}] dept={d.metadata.get("department")} | {d.text[:60].strip()}...')

## Topic 28 — Metadata Enrichment (Transformation)

In [ ]:
from llama_index.core import SimpleDirectoryReader
import re

def enrich_metadata(docs):
    for doc in docs:
        text = doc.text
        doc.metadata.update({
            'word_count': str(len(text.split())),
            'has_financial_data': str(bool(re.search(r'\$[\d,]+', text))),
            'has_table': str('|---' in text),
            'content_length': ('short' if len(text.split()) < 200
                               else 'medium' if len(text.split()) < 1000
                               else 'long'),
        })
    return docs

raw_docs = SimpleDirectoryReader('data/level2_samples', required_exts=['.txt', '.md'], filename_as_id=True).load_data()
enriched = enrich_metadata(raw_docs)
print(f'Enriched {len(enriched)} documents:')
for d in enriched:
    print(f"  [{d.metadata.get('file_name')}] words={d.metadata.get('word_count')} | has_$={d.metadata.get('has_financial_data')} | length={d.metadata.get('content_length')}")

## Topic 31 — IngestionPipeline

In [ ]:
from llama_index.core import Document
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.ingestion import IngestionPipeline

sample_docs = [
    Document(text='TechNova was founded in 2018. NovaSense monitors financial transactions. Revenue grew 86% in 2024 reaching $28.7M ARR. We have 187 employees across 3 offices.', metadata={'source': 'overview.txt', 'department': 'Marketing'}),
    Document(text='NovaSense uses Apache Kafka for stream processing. Scoring P99 latency: 47ms. Uptime SLA: 99.7%. Supports SWIFT, ACH, SEPA formats.', metadata={'source': 'architecture.md', 'department': 'Engineering'}),
]

pipeline = IngestionPipeline(
    transformations=[SentenceSplitter(chunk_size=128, chunk_overlap=20)]
)
nodes = pipeline.run(documents=sample_docs)

print(f'Pipeline: {len(sample_docs)} docs → {len(nodes)} nodes')
for i, node in enumerate(nodes):
    print(f'  Node {i+1}: dept={node.metadata.get("department")} | {node.text.strip()[:70]}...')

## 🏆 Mini-Project — TechNova Knowledge Base (End-to-End)

In [ ]:
from llama_index.core import Document, SimpleDirectoryReader, VectorStoreIndex
import pandas as pd, json
from pathlib import Path
from datetime import datetime

DATA_DIR = Path('data/level2_samples')
all_docs = []

# 1. Text/Markdown
txt_docs = SimpleDirectoryReader(input_dir=DATA_DIR, required_exts=['.txt', '.md'],
    filename_as_id=True, file_metadata=lambda fp: {'company': 'TechNova Inc.', 'project': 'kb'}).load_data()
for d in txt_docs:
    fn = d.metadata.get('file_name', '')
    d.metadata['doc_type'] = 'company_info' if 'overview' in fn else 'product_docs' if 'product' in fn else 'general'
    d.metadata['department'] = 'Marketing' if 'overview' in fn else 'Product'
all_docs.extend(txt_docs)
print(f'✅ Text/Markdown: {len(txt_docs)} docs')

# 2. CSV
df = pd.read_csv(DATA_DIR / 'employees.csv')
emp_docs = [Document(text=f'Employee: {r["name"]}\nDept: {r["department"]}\nRole: {r["role"]}\nLocation: {r["location"]}',
    doc_id=f'employee-{r["employee_id"]}',
    metadata={'employee_id': r['employee_id'], 'department': r['department'], 'role': r['role'],
               'location': r['location'], 'doc_type': 'employee_record', 'source': 'employees.csv', 'company': 'TechNova Inc.'})
    for _, r in df.iterrows()]
all_docs.extend(emp_docs)
print(f'✅ CSV employees: {len(emp_docs)} docs')

# 3. JSON
with open(DATA_DIR / 'quarterly_report.json') as f: qd = json.load(f)
cm = {'source': 'quarterly_report.json', 'quarter': qd['quarter'], 'year': str(qd['year']), 'company': qd['company'], 'department': 'Finance'}
fin, met = qd['financials'], qd['key_metrics']
json_ds = [
    Document(text=f"Q4 2024 Revenue: ${fin['revenue']['total']:,} | EBITDA: ${fin['ebitda']:,} | Net Income: ${fin['net_income']:,} | Gross Margin: {fin['gross_margin_pct']}%",
             doc_id=f"technova-Q4-2024-financials", metadata={**cm, 'doc_type': 'financial_report'}),
    Document(text=f"Q4 2024 Metrics: {met['new_customers']} new customers | Churn: {met['churn_rate_pct']}% | NPS: {met['nps_score']} | ARR Growth: {met['arr_growth_pct']}%",
             doc_id=f"technova-Q4-2024-metrics", metadata={**cm, 'doc_type': 'business_metrics'}),
    Document(text='Q4 Highlights:\n' + '\n'.join(f'• {h}' for h in qd['highlights']),
             doc_id=f"technova-Q4-2024-highlights", metadata={**cm, 'doc_type': 'highlights'}),
]
all_docs.extend(json_ds)
print(f'✅ JSON sections : {len(json_ds)} docs')
print(f'\n📊 TOTAL: {len(all_docs)} documents loaded')
print(f'   Avg text: {sum(len(d.text) for d in all_docs)//len(all_docs)} chars')

In [ ]:
# Build and query the index
index = VectorStoreIndex.from_documents(all_docs, show_progress=True)
qe = index.as_query_engine(similarity_top_k=3)

questions = [
    'What was TechNova Q4 2024 revenue?',
    'Who works in the Engineering department?',
    'What is NovaSense and what does it cost?',
]
for q in questions:
    r = qe.query(q)
    print(f'Q: {q}')
    print(f'A: {r}')
    print(f'Sources: {[s.node.metadata.get("doc_type") for s in r.source_nodes]}')
    print()

---
## Level 2 Knowledge Checklist

- [ ] Loaded .txt and .md files with rich metadata
- [ ] Loaded CSV as row-per-Document
- [ ] Loaded JSON as section-per-Document
- [ ] Wrote a custom `BaseReader`
- [ ] Used `excluded_embed_metadata_keys` and `excluded_llm_metadata_keys`
- [ ] Used `IngestionPipeline`
- [ ] Completed the TechNova mini-project

---
**Say 'NEXT LEVEL' for Level 3 — Documents & Nodes Deep Dive** 🚀